In [1]:
from rule4ml.models.wrappers import MultiModelWrapper

est = MultiModelWrapper()
est.load_default_models()

# Common places wrappers store loaded predictors
for attr in ["models", "model_dict", "estimators", "predictors", "_models", "_estimators", "_predictors"]:
    if hasattr(est, attr):
        obj = getattr(est, attr)
        print(attr, type(obj))
        if isinstance(obj, dict):
            print("  keys:", list(obj.keys())[:30])
        elif isinstance(obj, (list, tuple)):
            print("  len:", len(obj))

2026-01-28 10:05:54.178756: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-28 10:05:54.178827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-28 10:05:54.179856: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-28 10:05:54.186920: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-28 10:05:55.268794: W tensorflow/compiler/tf2

_models <class 'dict'>
  keys: ['BRAM', 'DSP', 'FF', 'LUT', 'CYCLES', 'INTERVAL']


In [2]:
estimator = MultiModelWrapper()
estimator.load_default_models()

def describe_multimodelwrapper(est):
    # Try common attribute names for where loaded predictors are stored
    candidates = ["models", "model_dict", "estimators", "predictors", "_models", "_estimators", "_predictors"]
    for a in candidates:
        if hasattr(est, a):
            obj = getattr(est, a)
            print(a, type(obj))
            if isinstance(obj, dict):
                print("  n_keys:", len(obj))
                print("  sample keys:", list(obj.keys())[:20])
            elif isinstance(obj, (list, tuple)):
                print("  len:", len(obj))
            else:
                # Some wrappers store a nested object; just show repr
                print("  repr:", repr(obj)[:300])

describe_multimodelwrapper(estimator)

_models <class 'dict'>
  n_keys: 6
  sample keys: ['BRAM', 'DSP', 'FF', 'LUT', 'CYCLES', 'INTERVAL']


In [3]:
for k, m in estimator._models.items():
    print(k, type(m))

BRAM <class 'rule4ml.models.wrappers.TorchModelWrapper'>
DSP <class 'rule4ml.models.wrappers.TorchModelWrapper'>
FF <class 'rule4ml.models.wrappers.TorchModelWrapper'>
LUT <class 'rule4ml.models.wrappers.TorchModelWrapper'>
CYCLES <class 'rule4ml.models.wrappers.TorchModelWrapper'>
INTERVAL <class 'rule4ml.models.wrappers.TorchModelWrapper'>


In [4]:
for name, m in estimator._models.items():
    print("\n", name)
    print("model:", m.model)
    print("checkpoint:", getattr(m, "checkpoint_path", None))


 BRAM
model: TorchGNN(
  (global_embeddings): ModuleDict(
    (g0_strategy_embedding): Embedding(3, 16)
    (g1_board_embedding): Embedding(5, 32)
    (g2_hls4ml_version_embedding): Embedding(3, 8)
    (g3_vivado_version_embedding): Embedding(13, 8)
  )
  (sequential_embeddings): ModuleDict(
    (s0_layer_type_embedding): Embedding(18, 8)
  )
  (numerical_layers): ModuleDict(
    (g4_numerical_layer): Linear(in_features=40, out_features=16, bias=False)
    (g5_numerical_layer): Linear(in_features=16, out_features=16, bias=False)
  )
  (g_projection): Sequential(
    (0): Linear(in_features=80, out_features=22, bias=True)
    (1): ReLU()
  )
  (gconvs): ModuleList(
    (0): GINConv(nn=Sequential(
      (0): Linear(in_features=22, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    ))
  )
  (norms): ModuleList(
    (0): GraphNorm(32)
  )
  (jumping_knowledge): JumpingKnowledge(cat)
  (attention_pool): AttentionalAggregation(gat

In [5]:
print(estimator._models["BRAM"].model)

TorchGNN(
  (global_embeddings): ModuleDict(
    (g0_strategy_embedding): Embedding(3, 16)
    (g1_board_embedding): Embedding(5, 32)
    (g2_hls4ml_version_embedding): Embedding(3, 8)
    (g3_vivado_version_embedding): Embedding(13, 8)
  )
  (sequential_embeddings): ModuleDict(
    (s0_layer_type_embedding): Embedding(18, 8)
  )
  (numerical_layers): ModuleDict(
    (g4_numerical_layer): Linear(in_features=40, out_features=16, bias=False)
    (g5_numerical_layer): Linear(in_features=16, out_features=16, bias=False)
  )
  (g_projection): Sequential(
    (0): Linear(in_features=80, out_features=22, bias=True)
    (1): ReLU()
  )
  (gconvs): ModuleList(
    (0): GINConv(nn=Sequential(
      (0): Linear(in_features=22, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    ))
  )
  (norms): ModuleList(
    (0): GraphNorm(32)
  )
  (jumping_knowledge): JumpingKnowledge(cat)
  (attention_pool): AttentionalAggregation(gate_nn=Sequentia

In [ ]:
# GNN being used when calling the multi model wrapper

In [6]:
import torch

m = estimator._models["BRAM"].model
s = torch.cat([p.detach().flatten().cpu() for p in m.parameters() if p.requires_grad])
print("n_params:", s.numel(), "mean:", s.mean().item(), "std:", s.std().item())

n_params: 40280 mean: -0.002804975491017103 std: 0.20184633135795593


In [7]:
import inspect, rule4ml, os, pkgutil
import rule4ml.models.wrappers as w

print("wrappers.py:", inspect.getfile(w))
print("rule4ml package dir:", os.path.dirname(inspect.getfile(rule4ml)))

wrappers.py: /home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/wrappers.py
rule4ml package dir: /home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml


In [8]:
import glob, os, rule4ml, inspect

pkg_dir = os.path.dirname(inspect.getfile(rule4ml))
for ext in ["*.pt", "*.pth", "*.ckpt", "*.pkl", "*.joblib"]:
    hits = glob.glob(os.path.join(pkg_dir, "**", ext), recursive=True)
    if hits:
        print(ext, "->", len(hits))
        print("sample:", hits[:10])

*.pt -> 6
sample: ['/home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/weights/v2/gnn/BRAM.weights.pt', '/home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/weights/v2/gnn/CYCLES.weights.pt', '/home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/weights/v2/gnn/DSP.weights.pt', '/home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/weights/v2/gnn/FF.weights.pt', '/home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/weights/v2/gnn/INTERVAL.weights.pt', '/home/users/jdweitz/miniforge3/envs/rule4ml_update/lib/python3.10/site-packages/rule4ml/models/weights/v2/gnn/LUT.weights.pt']
